# LAB | Connecting Python to SQL

Objective: identify the customers who were active in both May and June 2005 in the Sakila database, and measure how their rental activity differed between the two months.

## 1. Connect Python to the Sakila database

In [1]:
import getpass

import pandas as pd
from sqlalchemy import create_engine, text

# Connection parameters
user = "root"
password = getpass.getpass("MySQL password: ")
host = "localhost"
port = 3306
database = "sakila"

# Create the SQLAlchemy engine (MySQL + PyMySQL driver)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

# Check the connection
with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT DATABASE();")).scalar())

MySQL password:  ········


Connected to: sakila


In [2]:
# Preview the rental table
pd.read_sql(text("SELECT * FROM rental LIMIT 5;"), engine)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [3]:
# Number of rentals per month, to identify the period covered by the data
pd.read_sql(text("""
    SELECT YEAR(rental_date)  AS year,
           MONTH(rental_date) AS month,
           COUNT(*)           AS nb_rentals
    FROM rental
    GROUP BY YEAR(rental_date), MONTH(rental_date)
    ORDER BY year, month;
"""), engine)

,year,month,nb_rentals
0,2005,5,1156
1,2005,6,2311
2,2005,7,6709
3,2005,8,5686
4,2006,2,182


## 2. `rentals_month`: retrieve the rentals of a given month and year

In [4]:
def rentals_month(engine, month, year):
    """
    Retrieve all rentals made during a given month and year from the Sakila database.

    Parameters
    ----------
    engine : sqlalchemy.engine.Engine
        Connection engine to the Sakila database.
    month : int
        Month number (1-12).
    year : int
        Year (e.g. 2005).

    Returns
    -------
    pandas.DataFrame
        All columns of the `rental` table for the selected month and year.
    """
    # Parameterized query: avoids SQL injection and string concatenation
    query = text("""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = :month
          AND YEAR(rental_date)  = :year;
    """)
    return pd.read_sql(query, con=engine, params={"month": int(month), "year": int(year)})

In [5]:
may_rentals = rentals_month(engine, 5, 2005)
june_rentals = rentals_month(engine, 6, 2005)

print("May 2005 :", may_rentals.shape)
print("June 2005:", june_rentals.shape)
may_rentals.head()

May 2005 : (1156, 7)
June 2005: (2311, 7)


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


## 3. `rental_count_month`: number of rentals per customer

In [6]:
def rental_count_month(rentals_df, month, year):
    """
    Count the number of rentals made by each customer in a given month and year.

    Parameters
    ----------
    rentals_df : pandas.DataFrame
        DataFrame returned by `rentals_month`.
    month : int
        Month number, used to name the new column.
    year : int
        Year, used to name the new column.

    Returns
    -------
    pandas.DataFrame
        Two columns: `customer_id` and `rentals_<MM>_<YYYY>` (e.g. `rentals_05_2005`).
    """
    column_name = f"rentals_{int(month):02d}_{int(year)}"

    return (
        rentals_df
        .groupby("customer_id")["rental_id"]
        .count()
        .rename(column_name)
        .reset_index()
    )

In [7]:
may_counts = rental_count_month(may_rentals, 5, 2005)
june_counts = rental_count_month(june_rentals, 6, 2005)

display(may_counts.head())
display(june_counts.head())

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


## 4. `compare_rentals`: compare the activity between two months

In [8]:
def compare_rentals(df_month1, df_month2):
    """
    Combine two monthly rental-count DataFrames and compute the difference between them.

    Only customers active in both months are kept (inner join on `customer_id`).

    Parameters
    ----------
    df_month1 : pandas.DataFrame
        Output of `rental_count_month` for the first (reference) month.
    df_month2 : pandas.DataFrame
        Output of `rental_count_month` for the second month.

    Returns
    -------
    pandas.DataFrame
        `customer_id`, the two rental-count columns and a `difference` column
        equal to the second month minus the first month.
    """
    # Name of the rental-count column in each DataFrame
    col1 = [col for col in df_month1.columns if col != "customer_id"][0]
    col2 = [col for col in df_month2.columns if col != "customer_id"][0]

    combined = df_month1.merge(df_month2, on="customer_id", how="inner")
    combined["difference"] = combined[col2] - combined[col1]

    return combined

In [9]:
comparison = compare_rentals(may_counts, june_counts)
comparison.head(10)

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1
5,7,5,5,0
6,8,1,3,2
7,9,3,2,-1
8,10,1,5,4
9,11,1,3,2


## 5. Results

In [10]:
print("Active customers in May      :", may_counts.shape[0])
print("Active customers in June     :", june_counts.shape[0])
print("Active customers in both     :", comparison.shape[0])
print()
print("Average rentals in May       :", round(comparison["rentals_05_2005"].mean(), 2))
print("Average rentals in June      :", round(comparison["rentals_06_2005"].mean(), 2))
print()
print("Customers with more rentals  :", (comparison["difference"] > 0).sum())
print("Customers with fewer rentals :", (comparison["difference"] < 0).sum())
print("Customers with no change     :", (comparison["difference"] == 0).sum())

Active customers in May      : 520
Active customers in June     : 590
Active customers in both     : 512

Average rentals in May       : 2.23
Average rentals in June      : 3.87

Customers with more rentals  : 358
Customers with fewer rentals : 92
Customers with no change     : 62


In [11]:
# Largest increases
comparison.sort_values("difference", ascending=False).head(5)

,customer_id,rentals_05_2005,rentals_06_2005,difference
386,454,1,10,9
389,457,1,9,8
178,213,1,9,8
248,295,1,9,8
23,27,1,8,7


In [12]:
# Largest decreases
comparison.sort_values("difference", ascending=True).head(5)

,customer_id,rentals_05_2005,rentals_06_2005,difference
173,207,6,1,-5
230,274,6,2,-4
210,250,5,1,-4
167,198,5,1,-4
509,596,6,2,-4
